# Linear mixed model GWAS

> **Important notes for this notebook**
>
> In `GWAS5`, we ran standard linear and logistic regression
> (`plink --linear` and `--logistic`), which required removing
> cryptically related individuals and tested one SNP at a time while
> treating the polygenic background across the rest of the genome as
> noise. In this tutorial, we introduce **Linear Mixed Model (LMM)** and
> **Whole-Genome Regression (WGR)** GWAS methods—focusing on
> **LDAK-KVIK** ([Hof et al.,
> 2024](https://doi.org/10.1038/s41588-024-01986-w))—which use the
> heritability models from `GWAS8` to boost statistical power, retain
> related individuals, and scale to biobank cohorts.
>
> ## Learning outcomes
>
> - **Understand** the assumptions and statistical differences between
>   standard single-SNP regression (`PLINK`), classical LMMs
>   (`GCTA`/`GEMMA`), and two-step biobank methods (`BOLT-LMM`,
>   `REGENIE`, and `LDAK-KVIK`)
> - **Explain** how SNP heritability ($h^2_{\text{SNP}}$) and
>   Leave-One-Chromosome-Out (`LOCO`) predictions increase association
>   power while avoiding proximal contamination
> - **Execute** a two-step mixed-model GWAS using `LDAK-KVIK`
>   (`--kvik-step1` and `--kvik-step2`) for quantitative traits
> - **Apply** Saddlepoint Approximation (SPA) in `LDAK-KVIK`
>   (`--binary YES`) for unbalanced case-control traits
> - **Compare** association power and calibration between `PLINK` and
>   `LDAK-KVIK` in `R`
>
> ## How to make this notebook work
>
> - In this notebook, we will use both `bash command line` and `R`
>   programming languages. Remember to **change the kernel** whenever
>   you transition from one language to the other
>   (`Kernel --> Change Kernel`) indicated by the languages’ images. We
>   will first run `Bash` commands.
>
> <img src="Images/bash.png" alt="Bash" width="80"> Choose the Bash
> kernel

> **Some theory**
>
> ### Intuition First: How does a Mixed Model boost GWAS power?
>
> Imagine you are trying to hear a **solo singer** ($X_j \beta_j$, the
> effect of a single SNP on **Chromosome 1**) over a **background
> choir** ($g$, the combined polygenic effect of **Chromosomes 2–22**).
> In standard single-SNP regression (`plink --linear` in `GWAS5`), the
> genetic effects of thousands of causal variants on **Chromosomes
> 2–22** are left unmodeled and thrown into the random error term
> $\epsilon$. That background choir acts like loud room noise, making it
> harder to hear the solo singer on Chromosome 1.
>
> Modern **Linear Mixed Models (LMMs)** and **Whole-Genome Regression**
> methods (like `LDAK-KVIK`) solve this in two intuitive steps: 1.
> **Step 1 (Subtract the background choir using Chromosomes 2–22)**:
> Using the trait’s SNP heritability ($h^2_{\text{SNP}}$ from `GWAS8`),
> the model jointly fits all SNPs on Chromosomes 2–22 to predict each
> person’s background genetic value ($\hat{g}^{(-1)}$, called a
> **Leave-One-Chromosome-Out** or **`LOCO`** prediction) and subtracts
> it from the phenotype. 2. **Step 2 (Test each solo singer SNP on
> Chromosome 1 against the quieted phenotype)**: Because the polygenic
> background noise from the rest of the genome has been removed, the
> residual variance is smaller and true causal variants on Chromosome 1
> stand out with much stronger test statistics! Moreover, because Step 1
> naturally accounts for shared family genetics (`GRM` $K$), **we no
> longer need to discard related individuals** as we did in `GWAS3`.
>
> ### Why move beyond standard single-SNP GWAS (`PLINK`)?
>
> In `GWAS5`, we tested each SNP $j$ individually using ordinary least
> squares (`plink --linear`):
>
> $$y = X_j \beta_j + Z \gamma + \epsilon, \quad \epsilon \sim \mathcal{N}(0, \sigma_e^2 I)$$
>
> where $y$ is the phenotype vector ($N \times 1$), $X_j$ is the
> genotype vector for SNP $j$, and $Z$ is the matrix of fixed covariates
> (such as sex and top principal components). This classical approach
> has three major limitations in modern cohorts:
>
> 1.  **Loss of sample size from relatedness filtering**: Because
>     $\epsilon \sim \mathcal{N}(0, \sigma_e^2 I)$ assumes independent
>     residuals, related individuals violate the model and inflate Type
>     I error. In `GWAS3`, we had to drop related individuals
>     ($\hat{\pi} > 0.125$). In biobanks like the UK Biobank (~30% with
>     a 3rd-degree or closer relative), discarding relatives throws away
>     massive statistical power.
> 2.  **Omitted polygenic background reduces power**: For a polygenic
>     trait with heritability $h^2_{\text{SNP}}$, thousands of causal
>     variants on other chromosomes contribute variance $\sigma_g^2$ to
>     $y$. Leaving this polygenic signal in the error term $\epsilon$
>     inflates residual variance from $\sigma_e^2$ to
>     $\sigma_g^2 + \sigma_e^2$, weakening test statistics.
> 3.  **Case-control imbalance in binary traits**: Standard logistic
>     regression (`plink --logistic`) relies on normal theory that
>     breaks down for rare variants when cases are much rarer than
>     controls (e.g., 1:50 case-control ratio), leading to severe
>     false-positive inflation.
>
> ### Linear Mixed Models (LMMs) and Whole-Genome Regression
>
> A **Linear Mixed Model (LMM)** explicitly includes a random polygenic
> effect $g$ alongside the fixed effect of the tested SNP $X_j$:
>
> $$y = X_j \beta_j + Z \gamma + g + \epsilon, \quad g \sim \mathcal{N}(0, \sigma_g^2 K), \quad \epsilon \sim \mathcal{N}(0, \sigma_e^2 I)$$
>
> Equivalently, we can view $g = \sum_{k \neq j} X_k b_k$ as a
> **whole-genome regression** model over all other SNPs, where the prior
> variance of $b_k$ is governed by the **SNP heritability
> ($h^2_{\text{SNP}}$)** and the **heritability model**
> ($\mathbb{E}[h_k^2] \propto [2 f_k(1-f_k)]^{1+\alpha} w_k$) we studied
> in `GWAS8`.
>
> By conditioning on the polygenic background $\hat{g}^{(-c)}$ predicted
> from all chromosomes except chromosome $c$
> (**Leave-One-Chromosome-Out, LOCO**), mixed-model GWAS: - **Increases
> statistical power**: Residual variance drops from $\text{Var}(y)$ to
> approximately $(1 - h^2_{\text{SNP}})\text{Var}(y)$, which increases
> $\chi^2$ test statistics roughly in proportion to
> $1 / (1 - h^2_{\text{SNP}})$. - **Avoids proximal contamination**:
> Excluding the chromosome of the tested SNP when predicting
> $\hat{g}^{(-c)}$ ensures the polygenic predictor does not absorb the
> signal of the tested SNP via linkage disequilibrium (LD).
>
> ### Comparing Biobank-Scale Mixed-Model Methods
>
> Classical LMMs (`GCTA-MLMA`, `GEMMA`) explicitly build and factorize
> the dense $N \times N$ Genetic Relatedness Matrix $K$, which costs
> $\mathcal{O}(N^2 M)$ memory and $\mathcal{O}(N^3)$ time—impossible
> when $N = 500\text{,000}$. Modern biobank methods replace matrix
> factorization with a **2-Step Whole-Genome Regression** strategy:
>
> | Method | Heritability Model ($\alpha$) | Step 1 Effect Prior ($b_k$) | Binary Trait Support | Computational Scaling |
> |---------------|---------------|---------------|---------------|---------------|
> | **PLINK** (`--linear`/`--logistic`) | None ($\sigma_g^2 = 0$) | None (single-SNP only) | Wald / Firth test | Fast, $\mathcal{O}(NM)$, but requires unrelateds & lower power |
> | **Classical LMM** (`GCTA`, `GEMMA`) | GCTA ($\alpha = -1$) | Infinitesimal Gaussian: $b_k \sim \mathcal{N}(0, \sigma_g^2/M)$ | Liability / linear approx. | Slow, $\mathcal{O}(N^2 M + N^3)$, exact GRM inversion |
> | **BOLT-LMM** | GCTA ($\alpha = -1$) | Mixture of two Gaussians (spike-and-slab) | Linear model only | $\mathcal{O}(N M^{1.5})$ via conjugate gradient |
> | **REGENIE** | GCTA ($\alpha = -1$) | Stacked block-wise Ridge regression (infinitesimal) | Firth / Saddlepoint (SPA) | Fast two-step block regression, $\mathcal{O}(NM)$ |
> | **LDAK-KVIK** | **LDAK ($\alpha = -0.25$)** | **Elastic Net / Mixture prior** (adapts to sparse & polygenic traits) | **Saddlepoint Approx. (SPA)** | **Very fast & low RAM**, $\mathcal{O}(NM)$ randomized solver |
>
> ### How `LDAK-KVIK` Works (Step 1 and Step 2)
>
> `LDAK-KVIK` ([Hof et al.,
> 2024](https://doi.org/10.1038/s41588-024-01986-w)) divides mixed-model
> association testing into two fast steps:
>
> 1.  **Step 1 (`--kvik-step1`) — Heritability & LOCO Polygenic
>     Prediction**:
>     - Estimates SNP heritability ($\hat{h}^2_{\text{SNP}}$) under the
>       realistic LDAK-Thin model ($\alpha = -0.25$) using a randomized
>       Haseman-Elston / REML solver without forming an $N \times N$
>       GRM.
>     - Uses $\hat{h}^2_{\text{SNP}}$ to calibrate a variational Bayes
>       **Elastic Net** (mixture of Laplace and Gaussian priors) across
>       genome-wide SNPs, and computes 22 **Leave-One-Chromosome-Out
>       (LOCO)** polygenic predictions $\hat{g}^{(-c)}$
>       ($c = 1, \dots, 22$).
> 2.  **Step 2 (`--kvik-step2`) — Single-SNP Association Testing**:
>     - For each SNP $j$ on chromosome $c$, tests $X_j$ against the
>       LOCO-adjusted phenotype $y - Z\hat{\gamma} - \hat{g}^{(-c)}$
>       using a calibrated score test (or **Saddlepoint Approximation,
>       SPA**, when `--binary YES` is specified).
>
> > **Quick Concept Check (from Lecture 9)**: - *What happens if Step 1
> > accidentally includes all 22 chromosomes (including Chromosome 1)
> > when testing a causal SNP on Chromosome 1?* Its signal is **wiped
> > out** (*Proximal Contamination* — we subtracted the Solo Singer
> > along with the Background Choir!), which is why **`LOCO`** predicts
> > $\hat{g}^{(-1)}$ from Chromosomes 2–22 only. - *Why can an LMM
> > safely keep siblings and cousins in the study?* Because the
> > polygenic term $g \sim \mathcal{N}(0, \sigma_g^2 K)$ already models
> > shared family genetics!

# Two-Step Mixed-Model GWAS with `LDAK-KVIK`

### Preparing Phenotype and Covariates

<img src="Images/bash.png" alt="Bash" width="40"> Switch to the Bash
kernel.

First, let’s link the `Data` folder, create our working directory
`Results/GWAS9`, and format the phenotype (`Data/EUR.height`) and
covariate (`Data/EUR.covariate`) files for `LDAK` (which expects
headerless files where the first two columns are `FID` and `IID`):

In [ ]:
ln -sf ../Data
mkdir -p Results/GWAS9

# Strip headers from EUR.height and EUR.covariate for LDAK
awk 'NR>1 {print $1, $2, $3}' Data/EUR.height > Results/GWAS9/EUR.pheno
awk 'NR>1 {print $0}' Data/EUR.covariate > Results/GWAS9/EUR.covar

echo "Formatted individuals: $(wc -l < Results/GWAS9/EUR.pheno)"

### Running `LDAK-KVIK` Step 1 and Step 2 on Quantitative Traits

Now we run the two steps of `LDAK-KVIK` on our quality-controlled
European dataset (`Data/EUR.QC`).

Key `LDAK` flags used below: - `--kvik-step1 <prefix>`: Estimates SNP
heritability, fits the Step 1 Elastic Net whole-genome regression model
adjusting for covariates (`--covar`), and saves the 22
Leave-One-Chromosome-Out (`LOCO`) polygenic predictions
(`<prefix>.step1.loco.prs`). - `--kvik-step2 <prefix>`: Reads the Step 1
`LOCO` predictions from `<prefix>`, tests each SNP in `--bfile` against
the residualized phenotype, and writes the GWAS summary statistics to
`<prefix>.step2.assoc` (and `<prefix>.step2.summaries`). -
`--bfile <prefix>`, `--pheno <file>`, `--covar <file>`: Specify the
PLINK binary fileset, phenotype file, and covariate file (just as in
`PLINK`).

In [ ]:
# Step 1: Fit Whole-Genome Regression and generate LOCO predictions
ldak \
  --kvik-step1 Results/GWAS9/kvik_height \
  --bfile Data/EUR.QC \
  --pheno Results/GWAS9/EUR.pheno \
  --covar Results/GWAS9/EUR.covar

# Step 2: Perform single-SNP association testing conditioned on LOCO predictions
ldak \
  --kvik-step2 Results/GWAS9/kvik_height \
  --bfile Data/EUR.QC \
  --pheno Results/GWAS9/EUR.pheno \
  --covar Results/GWAS9/EUR.covar

head -n 10 Results/GWAS9/kvik_height.step2.assoc

Let’s inspect the columns of `kvik_height.step2.assoc`: - `Chromosome`,
`Predictor` (SNP ID), `Basepair`, `A1` (effect allele), `A2` (other
allele) - `Wald_Stat`: The test statistic ($Z$-score) - `Wald_P`:
Two-sided $p$-value - `Effect`, `SD`: Estimated effect size
($\hat{\beta}_j$) and standard error of allele `A1` -
`Effect_Liability`: Effect size on the liability scale (when analyzing
binary traits)

# Benchmarking Power: `LDAK-KVIK` vs. Standard `PLINK` GWAS

When does a Linear Mixed Model provide the biggest gain over
`plink --linear`? Theory tells us the expected increase in test
statistics is proportional to $1 / (1 - h^2_{\text{SNP}})$: the higher
the heritability explained by the genome, the more background variance
Step 1 removes.

To see this clearly in our small teaching cohort ($N \approx 480$),
let’s use `ldak --make-phenos` to simulate a heritable polygenic trait
(`--her 0.7`, `--num-causals 200` causal SNPs, `--power -0.25`) directly
on `Data/EUR.QC` (which also records the true causal SNPs in
`sim_poly.effects`), and run both **standard `PLINK` linear regression**
and **`LDAK-KVIK`** on the exact same phenotype.

In [ ]:
# 1. Simulate a polygenic trait (h2 = 0.7, 200 causal variants) under the LDAK model
ldak \
  --make-phenos Results/GWAS9/sim_poly \
  --bfile Data/EUR.QC \
  --power -0.25 \
  --her 0.7 \
  --num-phenos 1 \
  --num-causals 200 \
  --random-seed 42

# 2. Run standard single-SNP linear regression in PLINK (with covariates)
plink \
  --bfile Data/EUR.QC \
  --pheno Results/GWAS9/sim_poly.pheno \
  --covar Data/EUR.covariate \
  --linear hide-covar \
  --out Results/GWAS9/plink_sim

# 3. Run LDAK-KVIK Step 1 and Step 2 on the same simulated phenotype
ldak \
  --kvik-step1 Results/GWAS9/kvik_sim \
  --bfile Data/EUR.QC \
  --pheno Results/GWAS9/sim_poly.pheno \
  --covar Results/GWAS9/EUR.covar

ldak \
  --kvik-step2 Results/GWAS9/kvik_sim \
  --bfile Data/EUR.QC \
  --pheno Results/GWAS9/sim_poly.pheno \
  --covar Results/GWAS9/EUR.covar

# Binary Traits and Saddlepoint Approximation (SPA)

When analyzing case-control phenotypes (coded as `1` = control, `2` =
case, or `0`/`1`), adding `--binary YES` tells `LDAK-KVIK` to use a
**Saddlepoint Approximation (SPA)** in Step 2.

Why is SPA essential? Standard score tests approximate the null
distribution of the test statistic using a normal distribution (which
only matches the mean and variance). When case-control ratios are
unbalanced (e.g., 500 cases vs. 50,000 controls) and a SNP has low minor
allele frequency, the true null distribution is heavily skewed, causing
standard tests to produce thousands of false positives. **SPA**
approximates the entire cumulant generating function (capturing skewness
and higher-order moments), ensuring accurate $p$-values even for
low-frequency variants in unbalanced cohorts.

Let’s run `LDAK-KVIK` with `--binary YES` on the binary case-control
phenotype stored in column 6 of `Data/TOY_TARGET_DATA.fam`.

In [ ]:
# Extract binary phenotype (FID IID CaseControl) from TOY_TARGET_DATA.fam
awk '{print $1, $2, $6}' Data/TOY_TARGET_DATA.fam > Results/GWAS9/toy_binary.pheno

# Step 1 with --binary YES
ldak \
  --kvik-step1 Results/GWAS9/kvik_binary \
  --bfile Data/TOY_TARGET_DATA \
  --pheno Results/GWAS9/toy_binary.pheno \
  --binary YES

# Step 2 with --binary YES (activates Saddlepoint Approximation)
ldak \
  --kvik-step2 Results/GWAS9/kvik_binary \
  --bfile Data/TOY_TARGET_DATA \
  --pheno Results/GWAS9/toy_binary.pheno \
  --binary YES

head -n 5 Results/GWAS9/kvik_binary.step2.assoc

### Visualizing and Comparing `LDAK-KVIK` vs. `PLINK` in R

<img src="Images/R.png" alt="R" width="40"> Switch to the R kernel.

Let’s switch to the **R kernel** to visualize the `LDAK-KVIK` Manhattan
and Q-Q plots and compare the test statistics at the true causal
variants (`sim_poly.effects`) between `LDAK-KVIK` and `PLINK`.

In [ ]:
# Setup to avoid long messages and plot on screen
options(warn = -1)
options(jupyter.plot_mimetypes = 'image/png')
options(repr.plot.width = 10, repr.plot.height = 4.5)

suppressMessages(suppressWarnings(library(ggplot2)))
suppressMessages(suppressWarnings(library(qqman)))

# Load PLINK and LDAK-KVIK results for the simulated polygenic trait
plink_res <- read.table("Results/GWAS9/plink_sim.assoc.linear", header = TRUE)
kvik_res  <- read.table("Results/GWAS9/kvik_sim.step2.assoc", header = TRUE)
causals   <- read.table("Results/GWAS9/sim_poly.effects", header = TRUE)

# Merge by SNP identifier
merged <- merge(plink_res[, c("SNP", "CHR", "BP", "STAT", "P")],
                kvik_res[, c("Predictor", "Wald_Stat", "Wald_P", "Effect", "SD")],
                by.x = "SNP", by.y = "Predictor")

merged$IsCausal <- ifelse(merged$SNP %in% causals$Predictor, "Causal SNP", "Null SNP")
merged$CHISQ_PLINK <- merged$STAT^2
merged$CHISQ_KVIK  <- merged$Wald_Stat^2

cat("Mean Chi-Square at True Causal SNPs (PLINK):    ", round(mean(merged$CHISQ_PLINK[merged$IsCausal == "Causal SNP"], na.rm = TRUE), 3), "\n")
cat("Mean Chi-Square at True Causal SNPs (LDAK-KVIK):", round(mean(merged$CHISQ_KVIK[merged$IsCausal == "Causal SNP"], na.rm = TRUE), 3), "\n")

In [ ]:
# Manhattan and Q-Q plots for LDAK-KVIK
par(mfrow = c(1, 2))
manhattan(kvik_res,
          chr = "Chromosome", bp = "Basepair", p = "Wald_P", snp = "Predictor",
          highlight = causals$Predictor,
          col = c("steelblue", "tomato"),
          main = "LDAK-KVIK Manhattan Plot")

qq(kvik_res$Wald_P, main = "LDAK-KVIK Q-Q Plot", col = "steelblue", pch = 19)

> **Stop - Read - Solve**
>
> - **Q1. Create a scatter plot comparing `-log10(P)` from `PLINK` on
>   the x-axis against `-log10(Wald_P)` from `LDAK-KVIK` on the y-axis,
>   coloring points by `IsCausal` (`"Causal SNP"` vs. `"Null SNP"`), and
>   add the $y = x$ identity line (`geom_abline`).**
> - **Q2. Compute the genomic inflation factor
>   $\lambda_{\text{GC}} = \text{median}(\chi^2) / \text{qchisq}(0.5, 1)$
>   for both `PLINK` and `LDAK-KVIK`. Why is it crucial that `LDAK-KVIK`
>   uses Leave-One-Chromosome-Out (`LOCO`) predictions in Step 1 rather
>   than predicting polygenic effects from all 22 chromosomes
>   simultaneously?**
>
> Hint: Use `merged` from the previous cell,
> `ggplot(merged, aes(x = -log10(P), y = -log10(Wald_P), color = IsCausal))`,
> and `median(merged$CHISQ_KVIK, na.rm = TRUE) / qchisq(0.5, 1)`.

In [ ]:
# Write your code here


> **Solution**
>
> In `R`, we plot `-log10(P)` from `PLINK` against `-log10(Wald_P)` from
> `LDAK-KVIK` and compute $\lambda_{\text{GC}}$ for both methods:
>
> ``` bash
> # Q1. Scatter plot of -log10(p) values: PLINK vs. LDAK-KVIK
> p_comp <- ggplot(merged, aes(x = -log10(P), y = -log10(Wald_P), color = IsCausal, alpha = IsCausal)) +
>   geom_point(size = 1.8) +
>   geom_abline(intercept = 0, slope = 1, linetype = "dashed", color = "black") +
>   scale_color_manual(values = c("Causal SNP" = "tomato", "Null SNP" = "steelblue")) +
>   scale_alpha_manual(values = c("Causal SNP" = 0.9, "Null SNP" = 0.25)) +
>   theme_bw(base_size = 13) +
>   labs(
>     title = "Association Power: Standard PLINK vs. LDAK-KVIK",
>     x = expression(-log[10](italic(p)) ~ "(PLINK Linear Regression)"),
>     y = expression(-log[10](italic(p)) ~ "(LDAK-KVIK Mixed Model)"),
>     color = "Variant Class"
>   ) +
>   guides(alpha = "none")
>
> show(p_comp)
>
> # Q2. Genomic inflation factor (lambda_GC)
> lambda_plink <- median(merged$CHISQ_PLINK, na.rm = TRUE) / qchisq(0.5, df = 1)
> lambda_kvik  <- median(merged$CHISQ_KVIK,  na.rm = TRUE) / qchisq(0.5, df = 1)
>
> cat("Lambda_GC (PLINK):    ", round(lambda_plink, 4), "\n")
> cat("Lambda_GC (LDAK-KVIK):", round(lambda_kvik, 4), "\n")
> ```
>
> - **Q1**: Notice how the true causal SNPs (red points) lie
>   systematically above the dashed $y = x$ identity line, showing that
>   conditioning on the Step 1 polygenic predictor increases association
>   test statistics.
> - **Q2**: If Step 1 included SNPs on the **same chromosome** as the
>   SNP being tested in Step 2, the whole-genome regression predictor
>   $\hat{g}$ would already capture part of the effect of the tested SNP
>   (both directly and through LD with nearby tagging SNPs). Subtracting
>   $\hat{g}$ from $y$ would therefore subtract out the true association
>   signal—a phenomenon called **proximal contamination**—causing a
>   severe loss of statistical power. Using **Leave-One-Chromosome-Out
>   (`LOCO`)** predictions $\hat{g}_{-c}$ removes polygenic variance
>   from the other 21 chromosomes while leaving all genetic signal on
>   chromosome $c$ intact for Step 2 testing.

------------------------------------------------------------------------

### Conclusion

In this tutorial, we have learned how Linear Mixed Model and
Whole-Genome Regression GWAS methods (`LDAK-KVIK`) leverage SNP
heritability ($h^2_{\text{SNP}}$ from `GWAS8`) and `LOCO` polygenic
predictions to increase discovery power and control confounding across
quantitative and binary traits. Because Step 2 of an LMM GWAS still
tests one SNP at a time on the `LOCO` residuals, the resulting GWAS
summary statistics (`*.step2.assoc` and `*.step2.summaries`) remain
blurred by local Linkage Disequilibrium (LD) on each chromosome. In the
next two sessions, we will see how pairing **GWAS summary statistics**
with an **external LD reference panel** allows us to deconvolve LD
locally to pinpoint causal variants (**Fine-Mapping in `GWAS10`**) and
globally to build genome-wide predictors (**Advanced PRS in `GWAS11`**).

> **Other take-home messages**
>
> - **Two-Step Whole-Genome Regression Scales LMMs to Biobanks**:
>   Instead of inverting an $N \times N$ kinship matrix
>   ($\mathcal{O}(N^3)$), methods like `LDAK-KVIK`, `REGENIE`, and
>   `BOLT-LMM` predict the polygenic background in Step 1 (`LOCO`) and
>   test individual SNPs against the residuals in Step 2
>   ($\mathcal{O}(NM)$).
> - **Realistic Heritability Priors Improve Step 1**: Because
>   `LDAK-KVIK` combines the LDAK heritability model ($\alpha = -0.25$)
>   with a flexible Elastic Net prior, its Step 1 polygenic predictions
>   adapt accurately to both highly polygenic traits and traits with
>   large-effect loci.
> - **Always Use SPA for Unbalanced Binary Traits**: When analyzing
>   case-control traits with low case prevalence, enabling
>   `--binary YES` activates the Saddlepoint Approximation (SPA),
>   preventing false-positive inflation at low-frequency variants.